# 05A 从 Python 函数到 MCP 工具

第二节我们已经能直接调用文件函数。现在换一个条件：工具在另一个进程里，应用怎样知道它有什么能力、需要什么参数、返回什么？

本节先区分 **MCP 连接**与 **Agent Skills**。MCP 负责标准化交换；Skill 通过 `SKILL.md` 记录适用场景、步骤和可选资源。二者可以一起使用，也可以独立使用。

先按课程单元 README 安装 `requirements.txt`。本实验实际启动随课只读服务端，不需要模型密钥或网络。

**贯穿案例**

任务是回答教学仓库的 `main.py` 怎样调用 `config.py`，而不是让模型凭文件名猜。Notebook 通过本机 MCP 服务列出可读文件、实际读取带行号的源码，同时用 `../outside.txt` 验证越界请求被拒绝；最后保存协议请求、响应和权限检查的审计记录。


## 学习路线与配套课件

实践 5A：检查 MCP 连接并输出能力审计（`L05-P01`）。

建议先完成本节概念正课，再进入本实践小节。这个 Notebook 可用独立新内核从头运行。先预测，再执行代码、修改一个条件并解释结果。

对应课件稳定编号：L05.02-S01、L05.02-S03、L05.02-S02、L05.04-S02、L05.04-S05。页数调整不改变这些编号。

按“问题—代码—观察—练习—可复用结果”的顺序学习。先完成练习，再展开参考分析。回放、保存的真实记录和新的在线请求都会明确标注；在线请求默认关闭。

In [1]:
from pathlib import Path
import sys, json, asyncio, importlib.metadata, hashlib, os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
CHAPTER = ROOT / "05 MCP 与 Skills 能力扩展"
sys.path.insert(0, str(CHAPTER / "代码"))
sys.path.insert(0, str(ROOT / "02 从大模型到 Agent 组成与最小循环/代码"))
print("MCP SDK:", importlib.metadata.version("mcp"))
print("课堂协议: 2026-07-28；本地 stdio；生成服务仍为 DeepSeek。")


MCP SDK: 2.1.1
课堂协议: 2026-07-28；本地 stdio；生成服务仍为 DeepSeek。


## 1. 谁在做什么

- **Host（宿主应用）**：我们的学习助手，组织用户任务、模型、Skill 与工具权限。
- **Client（客户端）**：应用中的 MCP 连接组件，发送发现和调用请求。
- **Server（服务端）**：暴露工具的进程，本课只读取第二节自编仓库。

DeepSeek 返回的工具调用不是 MCP 请求。应用需要把模型调用映射到 MCP 客户端。服务器也不因为使用了 MCP 就自动拥有一个大模型。

服务可以提供 tools、resources、prompts；本节跑通 tools，不把文件工具与协议中的 resource 概念混为一谈。


## 2. 先固定版本，再读协议

本课实测官方 Python SDK **2.1.1**，使用 **2026-07-28** 规范。当前版本采用请求携带元信息，旧版 `initialize → initialized` 握手不再是此版本的前置流程。

我们显式指定版本，观察 `tools/list` 和 `tools/call`。SDK 的 `auto` 模式另有发现探测和旧版本兼容；不要把一次实验的消息序列说成所有版本相同。

stdio 把标准输入/输出用于协议，服务端日志应走 stderr。子进程只是独立进程，**不等于沙箱**。本课真实边界来自文件可读清单。


In [2]:
from mcp_connection import probe
# await 适用于 Jupyter；普通 .py 入口使用 asyncio.run(probe())。
probe_result = await probe()
requests = [e["message"] for e in probe_result["events"] if e["direction"] == "client_to_server"]
print("实际发送的方法:", [m["method"] for m in requests])
print(json.dumps(requests[0], ensure_ascii=False, indent=2))
# JSON-RPC id 关联此层请求与响应；它不是模型的 tool_call_id。
responses = [e["message"] for e in probe_result["events"] if e["direction"] == "server_to_client"]
assert {m["id"] for m in requests} == {m["id"] for m in responses}


实际发送的方法: ['tools/list', 'tools/call', 'tools/call', 'tools/call']
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "tools/list",
  "params": {
    "_meta": {
      "io.modelcontextprotocol/protocolVersion": "2026-07-28",
      "io.modelcontextprotocol/clientInfo": {
        "name": "mcp",
        "version": "0.1.0"
      },
      "io.modelcontextprotocol/clientCapabilities": {}
    }
  }
}


### 沿 JSON-RPC id 找到真正的响应

配套课件：L05.02-S01、L05.02-S02。

本单元的 probe 已启动本机独立 MCP 服务进程。现在从它刚产生的记录中取请求和响应，按 JSON-RPC id 关联方法与结果。该 id 属于协议层；第二节的 tool_call_id 属于模型工具消息层，不能混为一套编号。

In [3]:
case_responses = {m["id"]: m for m in responses}
for case_request in requests:
    case_response = case_responses[case_request["id"]]
    print("请求 id:", case_request["id"], "方法:", case_request["method"],
          "响应字段:", [key for key in case_response if key not in {"id", "jsonrpc"}])
    assert ("result" in case_response) != ("error" in case_response)
assert any(m["method"] == "tools/list" for m in requests)
assert any(m["method"] == "tools/call" for m in requests)


请求 id: 1 方法: tools/list 响应字段: ['result']
请求 id: 2 方法: tools/call 响应字段: ['result']
请求 id: 3 方法: tools/call 响应字段: ['result']
请求 id: 4 方法: tools/call 响应字段: ['result']


**结果解读**

发现工具与调用工具是不同方法。收到 JSON-RPC result 也不等于工具成功：正常协议响应中仍可携带工具的 isError；随后会用越界读取验证。这里观察当前课程固定 SDK/协议的实测消息。

**小练习**

越界读取返回的是协议层 error 还是带 isError 的工具结果？在 probe_result["denied"] 中找证据。

<details><summary>完成后展开参考分析</summary>

本实验的越界读取是工具执行失败，响应作为工具结果传回；应读取 isError 和内容，不只看有无 JSON-RPC result。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

## 3. 服务端如何声明能力

下面展示并定义与 `代码/stdio_server.py` 一致的两个工具。装饰器注册函数；类型注解让 SDK 生成参数结构；函数体复用第二节的真实文件范围检查。

本单元创建的是 Notebook 进程中的示例对象，方便看懂声明。后面客户端实际启动 `.py` 文件中的服务；若修改服务端行为，请修改那个文件再重连，单改 Notebook 对象不会改变另一个进程。


In [4]:
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from agent_lab import RepoTools
repo = RepoTools(ROOT / "02 从大模型到 Agent 组成与最小循环/数据/demo-repo")
server = MCPServer("course-repository", version="1.0", log_level="ERROR")
@server.tool()
def list_files() -> dict:
    """列出当前教学仓库可读取的相对文件路径。"""
    return repo.list_files()

@server.tool()
def read_file(path: str) -> dict:
    """读取 list_files 返回的一个文本文件，返回原文和行号。"""
    if not path.strip() or len(path) > 200:
        raise ToolError('path 必须是 1–200 字符的非空相对路径')
    try:
        return repo.read_file(path)
    except PermissionError:
        raise ToolError('out_of_scope：只能读取 list_files 列出的课程文件') from None
    except (FileNotFoundError, ValueError, OSError):
        raise ToolError('read_failed：文件不存在、过大或无法读取') from None


In [5]:
# 看同一份声明如何变成工具 schema，不手写第二份参数表。
for tool in await server.list_tools():
    print(json.dumps(tool.model_dump(by_alias=True, exclude_none=True), ensure_ascii=False, indent=2))


{
  "name": "list_files",
  "description": "列出当前教学仓库可读取的相对文件路径。",
  "inputSchema": {
    "properties": {},
    "title": "list_filesArguments",
    "type": "object"
  }
}
{
  "name": "read_file",
  "description": "读取 list_files 返回的一个文本文件，返回原文和行号。",
  "inputSchema": {
    "properties": {
      "path": {
        "title": "Path",
        "type": "string"
      }
    },
    "required": [
      "path"
    ],
    "title": "read_fileArguments",
    "type": "object"
  }
}


## 4. 客户端发现、调用、读取观察

`connect()` 使用官方 `Client` 与 `stdio_client` 启动随课服务端，并被动记录协议消息；记录器不替代 MCP 协议实现。完整连接与记录代码位于 `代码/mcp_connection.py`，下面先展示连接入口，再做真实调用。

`async with` 保持连接在代码块内有效，退出时释放进程和流。`await` 等待操作完成。这里顺序调用即可，不要求先掌握并发编程。


In [6]:
from contextlib import asynccontextmanager
from mcp import Client, StdioServerParameters
from mcp_connection import recording_stdio
# 连接函数的实际实现；__file__ 在 Notebook 不存在，明确使用服务脚本路径。
@asynccontextmanager
async def lesson_connect():
    params = StdioServerParameters(command=sys.executable,
        args=[str(CHAPTER / "代码/stdio_server.py")], env={})
    async with Client(recording_stdio(params, []), mode="2026-07-28",
                      read_timeout_seconds=10, cache=None) as client:
        yield client


In [7]:
async with lesson_connect() as client:
    discovered = await client.list_tools()
    print("发现工具:", [t.name for t in discovered.tools])
    files = await client.call_tool("list_files", {})
    result = await client.call_tool("read_file", {"path": "config.py"})
    print("工具是否出错:", result.is_error)
    evidence = json.loads(result.content[0].text)
    for line in evidence["lines"]:
        print(f"{evidence['path']}:{line['line']} {line['text']}")


发现工具: ['list_files', 'read_file']
工具是否出错: False
config.py:1 """配置读取；本仓库只作为 Agent 阅读资料。"""
config.py:2 import json
config.py:3 import os
config.py:4 from pathlib import Path
config.py:5 
config.py:6 
config.py:7 def load_settings():
config.py:8     path = Path(os.environ.get("STUDYBOX_CONFIG", "settings.json"))
config.py:9     if not path.exists():
config.py:10         return {"course": "大模型应用开发", "timeout": 30}
config.py:11     return json.loads(path.read_text(encoding="utf-8"))


### 用真实 MCP 声明验证参数，再观察执行权限

配套课件：L05.02-S03、L05.04-S02、L05.04-S05。

客户端在上一单元从真实本地 MCP Server 发现工具并读回 config.py。这里用 jsonschema 在客户端按服务声明的 inputSchema 检查两组参数，同时复核上一单元保存的读取结果；本单元没有再次调用 read_file。服务端授权要由下一单元的越界调用检验，schema 通过不能授予文件读取权限。

In [8]:
from jsonschema import validate, ValidationError
case_read = next(tool for tool in discovered.tools if tool.name == "read_file")
case_schema = case_read.input_schema
validate({"path": "config.py"}, case_schema)
try:
    validate({"path": 42}, case_schema)
except ValidationError as case_error:
    print("客户端 schema 校验发现类型错误:", case_error.message)
else:
    raise AssertionError("path 类型错误应被拒绝")
print("上一单元 MCP 读取结果:", evidence["path"], "文件行数:", len(evidence["lines"]))
assert evidence["path"] == "config.py" and not result.is_error


客户端 schema 校验发现类型错误: 42 is not of type 'string'
上一单元 MCP 读取结果: config.py 文件行数: 11


**结果解读**

本单元的客户端校验发现参数类型错误；复核的 11 行来自上一单元真实的本地 MCP 调用，包含空行。本单元没有向服务端发送错误类型参数。这两项都不代表任意路径获准读取，下一步检查服务端实际拒绝结果。

**小练习**

如果把 path 改成 ../outside.txt，它能通过 schema 吗？通过后服务器应怎样处理？

<details><summary>完成后展开参考分析</summary>

它仍是字符串，通常会通过这个 schema；路径范围必须由服务端校验，并把拒绝作为真实工具结果返回。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

## 5. 能发现工具，不表示能读取任意路径

JSON 格式有效，只说明请求能被解析。服务端仍需检查文件范围。我们发一个越界路径，看收到的是正常文件内容还是带 `isError` 的失败结果。

协议级错误、工具执行失败、模型答案错误是不同层面。本例的预期拒绝由 SDK 的 `ToolError` 返回成工具失败；不要把 `resultType=complete` 误读为业务成功。


In [9]:
async with lesson_connect() as client:
    denied = await client.call_tool("read_file", {"path": "../outside.txt"})
    print("is_error:", denied.is_error)
    print(denied.content[0].text)
    assert denied.is_error


is_error: True
Error executing tool read_file: out_of_scope：只能读取 list_files 列出的课程文件


## 练习：读取调用方

把读取路径换成 `main.py`，找出调用 `load_settings()` 的行号。只改一个参数，保持服务端和协议不变。思考：我们刚才验证了模型会选择正确文件吗？


In [10]:
path_to_read = "main.py"  # 也可先自行换成另一个已列出的文件
async with lesson_connect() as client:
    student_result = await client.call_tool("read_file", {"path": path_to_read})
print(student_result.content[0].text)


{
  "path": "main.py",
  "lines": [
    {
      "line": 1,
      "text": "\"\"\"StudyBox 运行入口。\"\"\""
    },
    {
      "line": 2,
      "text": "from config import load_settings"
    },
    {
      "line": 3,
      "text": ""
    },
    {
      "line": 4,
      "text": ""
    },
    {
      "line": 5,
      "text": "def main():"
    },
    {
      "line": 6,
      "text": "    settings = load_settings()"
    },
    {
      "line": 7,
      "text": "    print(f\"欢迎，{settings['course']}！\")"
    },
    {
      "line": 8,
      "text": ""
    },
    {
      "line": 9,
      "text": ""
    },
    {
      "line": 10,
      "text": "if __name__ == \"__main__\":"
    },
    {
      "line": 11,
      "text": "    main()"
    }
  ]
}


In [11]:
lines = json.loads(student_result.content[0].text)["lines"]
hits = [x for x in lines if "settings = load_settings()" in x["text"]]
assert len(hits) == 1 and hits[0]["line"] == 6
print("main.py:6 调用配置函数。这里只有程序指定调用，尚未验证模型选择。")


main.py:6 调用配置函数。这里只有程序指定调用，尚未验证模型选择。


## 6. 交付：MCP 能力审计

把“连上了”变成可复查的记录：保存当时的协议与 SDK 版本、发现到的工具及其输入结构、允许范围，再同时留下一个成功调用和一个被拒绝调用。

这份审计证明协议层和边界层的实际行为，不证明之后的模型一定会选对工具。


In [12]:
RUN_DIR = ROOT / ".local/student-runs/l05"
RUN_DIR.mkdir(parents=True, exist_ok=True)
wire_requests = [e["message"] for e in probe_result["events"]
                 if e.get("direction") == "client_to_server" and "message" in e]
wire_responses = [e["message"] for e in probe_result["events"]
                  if e.get("direction") == "server_to_client" and "message" in e]
tool_schemas = [{"name": item["name"], "description": item.get("description", ""),
                 "input_schema": item.get("inputSchema", {})}
                for item in probe_result["tools"]["tools"]]
mcp_audit = {
    "artifact": "L05-mcp-capability-audit",
    "transport": "local_stdio",
    "protocol_version": probe_result["protocol"],
    "sdk_version": probe_result["sdk"],
    "discovered_tools": tool_schemas,
    "runtime_boundary": {
        "allowed_repository": "02 从大模型到 Agent 组成与最小循环/数据/demo-repo",
        "read_only": True,
        "subprocess_is_not_a_sandbox": True,
    },
    "successful_call": probe_result["config"],
    "denied_call": probe_result["denied"],
    "jsonrpc_request_ids": [m.get("id") for m in wire_requests if "id" in m],
    "jsonrpc_response_ids": [m.get("id") for m in wire_responses if "id" in m],
    "human_review_required": ["tool_descriptions_match_behavior", "returned_evidence_supports_later_claims"],
}
audit_path = RUN_DIR / "mcp-capability-audit.json"
audit_path.write_text(json.dumps(mcp_audit, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
assert mcp_audit["jsonrpc_request_ids"] == mcp_audit["jsonrpc_response_ids"]
assert probe_result["denied"]["isError"] is True
print("已生成 MCP 能力审计:", audit_path.relative_to(ROOT))
print("工具:", [x["name"] for x in tool_schemas], "请求 ID:", mcp_audit["jsonrpc_request_ids"])


已生成 MCP 能力审计: .local\student-runs\l05\mcp-capability-audit.json
工具: ['list_files', 'read_file'] 请求 ID: [1, 2, 3, 4]


## 回顾

我们实际完成了跨进程工具发现、参数传递、结果读取和范围拒绝。Host、Client、Server 各有职责；模型工具调用与 MCP 消息之间需要应用适配。

本节还生成了 `mcp-capability-audit.json`，留下版本、工具结构、成功与拒绝调用及 JSON-RPC ID。

接下来不增加工具数量，而是给助教一个可复用的仓库导读 Skill。

来源：[官方服务端教程](https://modelcontextprotocol.io/docs/2026-07-28/develop/build-server)、[Python SDK](https://github.com/modelcontextprotocol/python-sdk)、[2026-07-28 规范更新](https://blog.modelcontextprotocol.io/posts/2026-07-28/)。
